# Valutazione RAG — Diagnostica

Notebook di analisi per la pipeline RAG dell'assistenza universitaria: legge i risultati
dell'ultimo run di `evaluate.py` (`datasets/eval/results.json`), i chunk indicizzati
(`datasets/chunked/docstore.json`) e i vettori in Qdrant, per diagnosticare dove la pipeline
perde qualità — retrieval, chunking o generazione.

In [ ]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registra la proiezione 3d

sys.path.insert(0, "../src")

# Palette validata (skill dataviz) — vedi references/palette.md
PALETTE = {
    "blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100",
    "magenta": "#e87ba4", "green": "#008300", "violet": "#4a3aa7", "red": "#e34948",
}
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"
SURFACE = "#fcfcfb"
GOOD = "#0ca30c"

BLUE_CMAP = LinearSegmentedColormap.from_list("seq_blue", SEQ_BLUE)
DIVERGING_CMAP = LinearSegmentedColormap.from_list("div_blue_red", [PALETTE["red"], "#f0efec", PALETTE["blue"]])

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": AXIS,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "grid.color": GRID,
    "grid.linewidth": 0.6,
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

## 1. Performance Generali e Diagnostica RAG

Basato su `datasets/eval/results.json` (output di `evaluate.py`, RAGAS su 81 casi).

In [ ]:
RESULTS_PATH = Path("../datasets/eval/results.json")

with open(RESULTS_PATH, encoding="utf-8") as f:
    results = json.load(f)

def category_df(block):
    return pd.DataFrame(block["per_case"]).set_index("id")

df_retrieving = category_df(results["retrieving"])
df_generative = category_df(results["generative"])
df_end_to_end = category_df(results["end_to_end"])

df = df_retrieving.join(df_generative).join(df_end_to_end)
df.head()

### 1.1 Scatter Plot — Context Recall vs Faithfulness

Isola il collo di bottiglia: asse X il retrieval (`context_recall`), asse Y la generazione
(`faithfulness`). Le linee tratteggiate marcano la mediana; i casi con un punteggio sotto 0.5
su un asse sono etichettati per ispezione diretta.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

x = df["context_recall"]
y = df["faithfulness"]

ax.scatter(x, y, s=60, color=PALETTE["blue"], alpha=0.75, edgecolor="white", linewidth=0.6, zorder=3)
# Soglia diagnostica fissa (0.5), non la mediana campionaria: con recall/faithfulness quasi
# sempre alti la mediana finirebbe in un angolo e i quadranti perderebbero senso.
ax.axvline(0.5, color=AXIS, linewidth=1, linestyle="--", zorder=1)
ax.axhline(0.5, color=AXIS, linewidth=1, linestyle="--", zorder=1)

ax.set_xlabel("Context Recall  (qualità del Retrieval)")
ax.set_ylabel("Faithfulness  (aderenza dell'LLM al contesto)")
ax.set_title("Diagnostica del collo di bottiglia: Retrieval vs Generazione")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)

ax.text(0.03, 0.03, "retrieval scarso\ne LLM inaffidabile", fontsize=7.5, color=INK_SECONDARY, ha="left", va="bottom")
ax.text(0.97, 0.28, "retrieval buono,\nLLM allucina", fontsize=7.5, color=INK_SECONDARY, ha="right", va="bottom")
ax.text(0.03, 0.97, "retrieval scarso,\nLLM comunque fedele", fontsize=7.5, color=INK_SECONDARY, ha="left", va="top")
ax.text(0.53, 0.53, "entrambi buoni", fontsize=7.5, color=INK_SECONDARY, ha="left", va="bottom")

# Più casi possono cadere sullo stesso punto (o vicinissimi): raggruppo le etichette per
# coordinata arrotondata invece di sovrapporle illeggibili una sopra l'altra.
low_points = df[(df["context_recall"] < 0.5) | (df["faithfulness"] < 0.5)]
label_groups = defaultdict(list)
for case_id, row in low_points.iterrows():
    key = (round(row["context_recall"], 2), round(row["faithfulness"], 2))
    label_groups[key].append(case_id)

for (x_val, y_val), ids in label_groups.items():
    label = ", ".join(ids) if len(ids) <= 3 else f"{ids[0]} +{len(ids) - 1} altri"
    ax.annotate(label, (x_val, y_val), fontsize=6.5, color=INK_SECONDARY,
                xytext=(5, -5), textcoords="offset points")

plt.tight_layout()
plt.show()

### 1.2 Radar Chart — le 5 metriche RAGAS principali

Panoramica sintetica: Context Precision, Context Recall, Faithfulness, Answer Relevance,
Answer Correctness (medie sugli 81 casi).

In [ ]:
radar_metrics = ["context_precision", "context_recall", "faithfulness", "answer_relevancy", "answer_correctness"]
radar_labels = ["Context\nPrecision", "Context\nRecall", "Faithfulness", "Answer\nRelevance", "Answer\nCorrectness"]
radar_values = [df[m].mean() for m in radar_metrics]

angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
values_closed = radar_values + radar_values[:1]
angles_closed = angles + angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.set_facecolor(SURFACE)

ax.plot(angles_closed, values_closed, color=PALETTE["blue"], linewidth=2, zorder=3)
ax.fill(angles_closed, values_closed, color=PALETTE["blue"], alpha=0.15, zorder=2)
ax.scatter(angles, radar_values, color=PALETTE["blue"], s=40, zorder=4, edgecolor="white", linewidth=0.6)

ax.set_xticks(angles)
ax.set_xticklabels(radar_labels, fontsize=9, color=INK)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=7, color=INK_MUTED)
ax.spines["polar"].set_color(AXIS)
ax.grid(color=GRID)

for angle, value in zip(angles, radar_values):
    ax.annotate(f"{value:.2f}", (angle, value), fontsize=8, color=INK_SECONDARY,
                xytext=(0, 8), textcoords="offset points", ha="center")

ax.set_title("Le 5 metriche RAGAS principali (media sui casi)", pad=20)
plt.tight_layout()
plt.show()

### 1.3 Box Plot delle metriche — varianza e casi limite

La media (radar chart sopra) nasconde outlier e allucinazioni isolate: il box plot mostra la
distribuzione completa. I pallini rossi sono outlier statistici (oltre 1.5×IQR).

In [ ]:
box_metrics = ["context_precision", "context_recall", "context_entity_recall",
               "faithfulness", "answer_relevancy", "answer_correctness", "answer_similarity"]
box_labels = ["Context\nPrecision", "Context\nRecall", "Context Entity\nRecall",
              "Faithfulness", "Answer\nRelevancy", "Answer\nCorrectness", "Answer\nSimilarity"]

box_data = [df[m].dropna().values for m in box_metrics]
nan_counts = [df[m].isna().sum() for m in box_metrics]

fig, ax = plt.subplots(figsize=(11, 6))

ax.boxplot(box_data, patch_artist=True, widths=0.5, showfliers=True,
           medianprops=dict(color=INK, linewidth=1.5),
           boxprops=dict(facecolor=PALETTE["blue"], alpha=0.25, edgecolor=PALETTE["blue"]),
           whiskerprops=dict(color=INK_SECONDARY),
           capprops=dict(color=INK_SECONDARY),
           flierprops=dict(marker="o", markerfacecolor=PALETTE["red"], markeredgecolor="white",
                            markersize=5, alpha=0.7))

ax.set_xticklabels(box_labels, fontsize=8.5)
ax.set_ylabel("Punteggio")
ax.set_ylim(-0.08, 1.05)
ax.set_title("Distribuzione delle metriche sugli 81 casi")

for i, n in enumerate(nan_counts, start=1):
    if n:
        ax.annotate(f"{n} NaN", (i, -0.05), fontsize=7, color=INK_MUTED, ha="center", va="top")

plt.tight_layout()
plt.show()

### 1.4 Bar Chart raggruppato — confronto A/B tra run

Confronta tutti i file `results*.json` presenti in `datasets/eval/` (es. baseline vs una nuova
versione del prompt o del modello). **Nota:** al momento esiste un solo run salvato, quindi il
grafico mostra una sola serie — tornerà utile per il confronto non appena ne verrà salvato un
secondo con un nome diverso (es. `results_qwen_v2.json`).

In [ ]:
RESULTS_DIR = Path("../datasets/eval")
run_files = sorted(RESULTS_DIR.glob("results*.json"))

def load_run_means(path):
    with open(path, encoding="utf-8") as f:
        r = json.load(f)
    means = {}
    for cat in ("retrieving", "generative", "end_to_end"):
        means.update(r[cat]["mean"])
    return means

runs_df = pd.DataFrame({p.stem: load_run_means(p) for p in run_files}).T

compare_metrics = [m for m in ["context_precision", "context_recall", "faithfulness",
                                "answer_relevancy", "answer_correctness"] if m in runs_df.columns]
compare_labels = ["Context\nPrecision", "Context\nRecall", "Faithfulness",
                   "Answer\nRelevancy", "Answer\nCorrectness"][:len(compare_metrics)]

fig, ax = plt.subplots(figsize=(10, 5.5))

n_runs = len(runs_df)
n_metrics = len(compare_metrics)
bar_width = 0.8 / max(n_runs, 1)
x = np.arange(n_metrics)
run_colors = [PALETTE["blue"], PALETTE["orange"], PALETTE["aqua"]]

for i, (run_name, row) in enumerate(runs_df.iterrows()):
    offsets = x + (i - (n_runs - 1) / 2) * bar_width
    ax.bar(offsets, row[compare_metrics].values, width=bar_width * 0.9,
           color=run_colors[i % len(run_colors)], label=run_name)

ax.set_xticks(x)
ax.set_xticklabels(compare_labels, fontsize=9)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Punteggio medio")
# fig.suptitle (livello figura) invece di ax.set_title, così la nota "solo 1 run" qui sotto
# (che vive nel titolo dell'asse) non ci si sovrappone.
fig.suptitle("Confronto A/B tra run di valutazione", fontsize=12, y=1.01)
if n_runs > 1:
    ax.legend(frameon=False, loc="lower right")
else:
    ax.set_title("Solo 1 run disponibile — salva un altro results_*.json per il confronto A/B",
                 fontsize=8.5, color=INK_MUTED, pad=10)

plt.tight_layout()
plt.show()

## 2. Analisi di Chunking e Vector Store (Qdrant)

Basato su `datasets/chunked/docstore.json` (722 chunk persistiti da `chunker.py`) e sui vettori
live in Qdrant (collezione `ateneo_docs`).

In [ ]:
DOCSTORE_PATH = Path("../datasets/chunked/docstore.json")

with open(DOCSTORE_PATH, encoding="utf-8") as f:
    docstore_raw = json.load(f)["docstore/data"]

# L'ordine di iterazione del dict riflette l'ordine di chunking (top-to-bottom nel documento),
# quindi raggruppare per source_path preservando l'ordine dà già la sequenza corretta dei chunk.
nodes = []
for node_id, payload in docstore_raw.items():
    d = payload["__data__"]
    meta = d["metadata"]
    nodes.append({
        "node_id": d["id_"],
        "text": d["text"],
        "source_path": meta.get("source_path"),
        "categoria": meta.get("categoria"),
        "corso": meta.get("corso"),
    })

nodes_df = pd.DataFrame(nodes).set_index("node_id")
print(f"{len(nodes_df)} chunk caricati da {nodes_df['source_path'].nunique()} documenti")

### 2.1 Istogramma di lunghezza dei chunk

Token contati con lo stesso tokenizer usato in chunking (`BAAI/bge-m3`), per confrontare la
distribuzione reale con il target di 512 token del `HybridChunker` ([chunker.py](../src/chunking/chunker.py)).

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
nodes_df["n_tokens"] = [len(tokenizer.encode(t, add_special_tokens=False)) for t in nodes_df["text"]]

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(nodes_df["n_tokens"], bins=30, color=PALETTE["blue"], edgecolor=SURFACE, linewidth=0.5)
ax.axvline(512, color=PALETTE["red"], linewidth=1.5, linestyle="--", label="target chunker (512 token)")
ax.set_xlabel("Token per chunk (tokenizer BGE-M3)")
ax.set_ylabel("Numero di chunk")
ax.set_title(f"Distribuzione lunghezza chunk (n={len(nodes_df)})")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

nodes_df["n_tokens"].describe()

### Caricamento modelli e vettori (condiviso da 2.2 e dalla Sezione 3)

Carica BGE-M3 + reranker (GPU B, come in `app.py`) e recupera tutti i vettori densi da Qdrant.

In [ ]:
from qdrant_client import QdrantClient

from config import settings
from embedding.embedder import build_embedding_model
from retrieval.hybrid_search import embed_query, search_candidates
from retrieval.reranker import build_reranker, rerank

embed_model = build_embedding_model(device_id=settings.embeddings_device_id)
reranker_model = build_reranker(device_id=settings.embeddings_device_id)
qdrant = QdrantClient(url=settings.qdrant_url, grpc_port=settings.qdrant_grpc_port, prefer_grpc=True)

all_points = []
offset = None
while True:
    points, offset = qdrant.scroll(
        collection_name=settings.qdrant_collection,
        with_payload=["categoria", "source_path"],
        with_vectors=["dense"],
        limit=256,
        offset=offset,
    )
    all_points.extend(points)
    if offset is None:
        break

vectors = np.array([p.vector["dense"] for p in all_points])
categorie = [p.payload.get("categoria") for p in all_points]
print(f"{len(vectors)} vettori recuperati da Qdrant")

### 2.2 Scatter 3D degli embedding (t-SNE)

`umap-learn` non è tra le dipendenze del progetto ([requirements.txt](../requirements.txt)):
uso `sklearn.manifold.TSNE` come sostituto (stessa idea — proiezione non lineare in bassa
dimensione — risultati non identici a UMAP). Se serve UMAP vero, va aggiunto `umap-learn` ai
requirements.

Includo anche una query di esempio dal set di valutazione: viene proiettata **insieme** ai chunk
(il t-SNE va rifittato sull'unione, non supporta l'inserimento di nuovi punti a posteriori) e i
suoi Top-5 chunk recuperati sono evidenziati con un anello verde.

In [ ]:
with open("../datasets/eval/qa_test_set.json", encoding="utf-8") as f:
    qa_test_set = json.load(f)

sample_case = qa_test_set[0]
query_text = sample_case["question"]

query_embedding = embed_query(query_text, embed_model)
query_vec = np.array(query_embedding["dense_vecs"][0])

candidates = search_candidates(qdrant, settings.qdrant_collection, query_embedding)
reranked = rerank(query_text, candidates, reranker_model, limit=5)
top_ids = {c.id for c, _score in reranked}

from sklearn.manifold import TSNE

joint_vectors = np.vstack([vectors, query_vec[None, :]])
tsne = TSNE(n_components=3, perplexity=30, random_state=42, init="pca")
joint_coords = tsne.fit_transform(joint_vectors)
chunk_coords, query_coord = joint_coords[:-1], joint_coords[-1]

is_retrieved = np.array([p.id in top_ids for p in all_points])

In [ ]:
fig = plt.figure(figsize=(8, 7))
fig.patch.set_facecolor(SURFACE)
ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(SURFACE)

color_map = {"regolamenti": PALETTE["blue"], "corsi": PALETTE["orange"]}
point_colors = [color_map.get(c, INK_MUTED) for c in categorie]

ax.scatter(chunk_coords[~is_retrieved, 0], chunk_coords[~is_retrieved, 1], chunk_coords[~is_retrieved, 2],
           c=np.array(point_colors)[~is_retrieved], s=16, alpha=0.55, edgecolor="none")
ax.scatter(chunk_coords[is_retrieved, 0], chunk_coords[is_retrieved, 1], chunk_coords[is_retrieved, 2],
           c=np.array(point_colors)[is_retrieved], s=70, alpha=0.95, edgecolor=GOOD, linewidth=1.6)
ax.scatter(*query_coord, c=PALETTE["violet"], marker="*", s=280, edgecolor="white", linewidth=0.8, zorder=5)

ax.set_xlabel("t-SNE 1", fontsize=8, color=INK_MUTED)
ax.set_ylabel("t-SNE 2", fontsize=8, color=INK_MUTED)
ax.set_zlabel("t-SNE 3", fontsize=8, color=INK_MUTED)
ax.set_title(f"Proiezione 3D dei chunk (BGE-M3, t-SNE)\nquery: \"{query_text[:60]}...\"", fontsize=10)

legend_handles = [
    Patch(facecolor=color_map["regolamenti"], label="regolamenti"),
    Patch(facecolor=color_map["corsi"], label="corsi"),
    Patch(facecolor=SURFACE, edgecolor=GOOD, linewidth=1.6, label="chunk recuperati (Top-5)"),
    plt.Line2D([0], [0], marker="*", color="none", markerfacecolor=PALETTE["violet"], markersize=14, label="query"),
]
ax.legend(handles=legend_handles, frameon=False, loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

### 2.3 Heatmap di similarità tra chunk adiacenti

Matrice di similarità coseno tra i chunk dello stesso documento, nell'ordine in cui compaiono
(non tra documenti diversi, che non è il confronto interessante per tarare l'overlap). Uso il
documento con più chunk come esempio rappresentativo.

In [ ]:
doc_groups = defaultdict(list)
for p, vec in zip(all_points, vectors):
    doc_groups[p.payload.get("source_path")].append(vec)

biggest_doc = max(doc_groups, key=lambda k: len(doc_groups[k]))
doc_vectors = np.array(doc_groups[biggest_doc])

normed = doc_vectors / np.linalg.norm(doc_vectors, axis=1, keepdims=True)
sim_matrix = normed @ normed.T

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap=BLUE_CMAP, vmin=0, vmax=1)
ax.set_title(f"Similarità coseno tra chunk consecutivi\n{Path(biggest_doc).stem} ({len(doc_vectors)} chunk)", fontsize=10)
ax.set_xlabel("Indice chunk (ordine nel documento)")
ax.set_ylabel("Indice chunk (ordine nel documento)")
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Similarità coseno")
plt.tight_layout()
plt.show()

## 3. Distribuzione del Retrieval e Top-K

Riusa i modelli già caricati in Sezione 2.

### 3.1 Curva di similarità score del Top-K

Per ognuno degli 81 casi di valutazione, calcola lo score del reranker per i primi 10 candidati
e ne fa la media per rank — per stabilire visivamente se `rerank_top_k=5` e la soglia
`score > 0.3` ([retriever.py](../src/retrieval/retriever.py)) sono ben calibrati o tagliano
troppo presto/tardi.

In [ ]:
TOP_K_ANALYZE = 10
rank_scores = {k: [] for k in range(1, TOP_K_ANALYZE + 1)}

for case in qa_test_set:
    q_embedding = embed_query(case["question"], embed_model)
    q_candidates = search_candidates(qdrant, settings.qdrant_collection, q_embedding)
    q_reranked = rerank(case["question"], q_candidates, reranker_model, limit=TOP_K_ANALYZE)
    for rank, (_candidate, score) in enumerate(q_reranked, start=1):
        rank_scores[rank].append(score)

ranks = list(rank_scores.keys())
means = [np.mean(rank_scores[r]) for r in ranks]
p25 = [np.percentile(rank_scores[r], 25) for r in ranks]
p75 = [np.percentile(rank_scores[r], 75) for r in ranks]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.fill_between(ranks, p25, p75, color=PALETTE["blue"], alpha=0.15, label="IQR (25°-75° percentile)")
ax.plot(ranks, means, color=PALETTE["blue"], linewidth=2, marker="o", markersize=5, label="score medio reranker")
ax.axhline(0.3, color=PALETTE["red"], linewidth=1.2, linestyle="--", label="soglia attuale (score > 0.3)")
ax.axvline(5, color=AXIS, linewidth=1, linestyle=":", label="rerank_top_k attuale (5)")

ax.set_xlabel("Rank (posizione dopo il reranking)")
ax.set_ylabel("Score del reranker")
ax.set_title(f"Score di similarità per rank (Top-{TOP_K_ANALYZE}, media su {len(qa_test_set)} query)")
ax.set_xticks(ranks)
ax.legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.show()

### 3.2 Heatmap di correlazione tra metriche

Correlazione di Pearson tra le metriche RAGAS calcolate per-caso (`df` dalla Sezione 1) — utile
per capire, ad esempio, se `context_precision` e `answer_relevancy` si muovono insieme.

In [ ]:
corr_metrics = ["context_precision", "context_recall", "context_entity_recall",
                "faithfulness", "answer_relevancy", "answer_correctness", "answer_similarity"]
corr_labels = ["Ctx\nPrecision", "Ctx\nRecall", "Ctx Entity\nRecall", "Faithfulness",
               "Answer\nRelevancy", "Answer\nCorrectness", "Answer\nSimilarity"]

corr = df[corr_metrics].corr(method="pearson")

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(corr.values, cmap=DIVERGING_CMAP, vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_labels)))
ax.set_yticks(range(len(corr_labels)))
ax.set_xticklabels(corr_labels, fontsize=8, rotation=45, ha="right")
ax.set_yticklabels(corr_labels, fontsize=8)

for i in range(len(corr_labels)):
    for j in range(len(corr_labels)):
        val = corr.values[i, j]
        text_color = "white" if abs(val) > 0.6 else INK
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7.5, color=text_color)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Correlazione di Pearson")
ax.set_title("Correlazione tra le metriche RAGAS (per-caso)")
plt.tight_layout()
plt.show()

## Note e limiti

- **t-SNE al posto di UMAP** (§2.2): `umap-learn` non è installato nel progetto; se serve la
  proiezione UMAP vera va aggiunto a `requirements.txt`.
- **Confronto A/B (§1.4)**: al momento esiste un solo `results.json` salvato — il grafico è
  pronto per confrontare più run non appena ce ne sarà un secondo.
- **`context_entity_recall`** ha NaN su 44/81 casi (§1.3): la metrica non è ben adatta a risposte
  fattuali corte (es. "157,04 €") dove l'estrattore di entità di RAGAS spesso non trova nulla.
- La query di esempio in §2.2 è fissa (`qa_test_set[0]`) — cambia `sample_case = qa_test_set[i]`
  per ispezionare un altro caso.